Theorems (or conjectures) for the theory of <a class="ProveItLink" href="theory.ipynb">proveit.physics.quantum.QEC2</a>
========

In [1]:
import proveit
# Prepare this notebook for defining the theorems of a theory:
%theorems_notebook # Keep this at the top following 'import proveit'.

from proveit            import b, c, e, m, n, s, A, B, X, ExprTuple
from proveit.logic      import Equals, Exists, Forall, InSet, Implies
from proveit.logic.sets import (Disjoint, Set, SetOfAll, SubsetEq,
                                SymmetricDifference, Union, Difference, Intersect)
from proveit.numbers    import (one, two, Add, frac, greater_eq,
                                LessEq, Mult, Natural, Neg)

from proveit.physics.quantum.QEC2 import (
        _ell, _max_buf_weight, BufiloSequences, BufiloSets, Errors,
        f_one_to_n, IrreducibleBufiloSets, m_prime, MalignantSets, State,
        States, Weight)


In [2]:
%begin theorems

Defining theorems for theory 'proveit.physics.quantum.QEC2'
Subsequent end-of-cell assignments will define theorems
'%end theorems' will finalize the definitions


#### Facts About Sets

In [3]:
difference_is_subset_of_symmetric_difference = (
    Forall((A, B),
           SubsetEq(Difference(A, B), SymmetricDifference(A, B))
    )
)

#### Facts About Weight

In [4]:
binary_disjoint_weight_additivity = (
    Forall((A, B),
           Equals(Weight(Union(A, B)),
                          Add(Weight(A), Weight(B))),
           conditions = [Disjoint(A, B)]
    )
)

In [5]:
weight_of_differences_inequality = (
    Forall((A, B),
           greater_eq(Weight(Difference(A, B)),
                      Weight(Difference(B, A))),
           conditions = [greater_eq(Weight(A), Weight(B))]
    )
)

#### Malignant Set, $m$

We let $\textrm{MALS}$ denote the set of all possible _malignant sets_ (of faults) of a minimum-weight decoder.

Generally, a malignant set $m \in \textrm{MALS}$ is any set of faults that causes (or can cause) the QEC system to experience a logical failure. In the language and notation of Beverland, _et al._ (2025), a malignant set $m$ is one such that $H(m + c) = 0$ while $A(m + c) \ne 0$, where $H \in \mathbb{F}_{2}^{M \times N}$ is the “check matrix”, $A \in \mathbb{F}_{2}^{K \times N}$ is the “action matrix”, and $c = \mathcal{C}(\sigma)$ is the correction provided by the minimum-weight decoding algorithm $\mathcal{C}$.

Considered as a _set_ (instead of a Beverland vector), a malignant set $m$ is characterized by the fact that, for any such $m$, there exists both a correction $c$ and BUFILO $b$ such that $w(c) \le w(m)$ and $m \Delta c = b$, as captured more formally in the following theorem:

In [6]:
mal_set_property = (
    Forall(m,
       Exists(c,
       Exists(b, Equals(SymmetricDifference(m, c), b),
              domain=IrreducibleBufiloSets),
       conditions=[LessEq(Weight(c), Weight(m))]),
    domain=MalignantSets)
)

#### Theorem (0)

##### For a minimum-weight decoder, every malignant fault set contains a non-minority subset of some BUFILO.

In [7]:
mal_set_contains_non_minority_subset_of_bufilo = Forall(m,
       Exists((m_prime, b),
       greater_eq(Weight(m_prime), Mult(frac(one, two), Weight(b))),
       conditions=[SubsetEq(m_prime, m), InSet(b, IrreducibleBufiloSets), SubsetEq(m_prime, b)]),
domain=MalignantSets)

#### Theorem (1)
##### The sets of faults generated by $\mathcal{F}_{\ell,w_{\text{BUF}}}^{\text{seq}}$ exactly matches the BUFILOs up to weight $w_{\text{BUF}}$.

NOTE: In the formulation below, the `ExprTuple` construct is not formatting correctly, which makes the tuple $(f_{1}, f_{2},\ldots, f_{n})$ _appear_ as individual items $f_{1}, f_{2}, \ldots, f_{n}$.

In [8]:
from proveit.physics.quantum.QEC2 import Faults
from proveit.logic.sets import UnionAll

In [9]:
# bufilos_from_buf_seqs = Forall(n,
#        Equals(SetOfAll(ExprTuple(f_one_to_n), Set(f_one_to_n),
#                        condition=InSet(ExprTuple(f_one_to_n), BufiloSequences),
#                        domain = Faults),
#               SetOfAll(b, b, conditions=[LessEq(Weight(b), _max_buf_weight)], domain=BufiloSets)),
# domain = Natural)

In [10]:
# Theorem (1)
bufilos_from_buf_seqs = Equals(
    # LHS
    UnionAll(n, SetOfAll(ExprTuple(f_one_to_n), Set(f_one_to_n),
                       condition=InSet(ExprTuple(f_one_to_n), BufiloSequences),
                       domain = Faults),
          domain = Natural),
    # RHS
    SetOfAll(b, b, conditions=[LessEq(Weight(b), _max_buf_weight)], domain=BufiloSets)
)

ALTERNATIVE approach, WW thinks is more promising and comprehensive:

For each $\nu: \mathcal{S} \rightarrow \mathcal{D}$ ($\nu$ maps syndromes to an element of the syndrome; it's a "choice function" of a sort --- (include the “choice function” characteristic as an extra condition)), and for each BUFILO $b$, there exists $n$ and $f_{1}, f_{2}, \ldots, f_{n}$ such that $\{f_{1}, f_{2}, \ldots, f_{n}\} = b$ and:

$\forall_{i \in \{1,\ldots, n-1\}}$:

(1) $A_{l} \{f_{1}, \ldots, f_{i}\} = 0 \,\land \{f_{i+1}, \ell\} = 0$

or (2) $A_{l} \{f_{1}, \ldots, f_{i}\} \ne 0 \,\land \nu(H \{f_{1}, \ldots, f_{i}\}) \in H \{f_{i+1}\}$

Putting this together, we have:

$\forall_{\nu:[\mathcal{S}\rightarrow\mathcal{D}] \,|\, \forall_{X}\big(\nu(X) \in X\big)} \forall_{b \in \text{BUF}}
\Big[
\exists_{f_{1}, f_{2}, \ldots, f_{n} \,|\, n \in \mathbb{N}}
\Big[$

$\Big(\{f_{1}, f_{2}, \ldots, f_{n}\} = b \big)
\,\land\,$

$\forall_{i \in \{1,\ldots, n-1\}}\Big[\big[A_{l} \{f_{1}, \ldots, f_{i}\} = 0 \,\land [f_{i+1}, \ell]_{+} = 0\big] \lor \big[A_{l} \{f_{1}, \ldots, f_{i}\} \ne 0 \,\land \nu(H \{f_{1}, \ldots, f_{i}\}) \in H \{f_{i+1}\}\big]\Big]$

$\Big]\Big]$

In [11]:
from proveit import f, i, n, A, S, D, X, Y, Function, IndexedVar
from proveit.logic import And, Exists, NotEquals, Or
from proveit.logic.sets import IsFunction
from proveit.numbers import zero, one, Interval, subtract
from proveit.linear_algebra import AntiCommutator
from proveit.physics.quantum.QEC2 import ActionFunction, CheckFunction, Detectors, _ell, f_one_to_i, nu, Syndromes

In [12]:
Forall(nu,
       Forall(b,
              Exists((f_one_to_n),
                     And(
                         Equals(Set(f_one_to_n), b),
                         Forall(i,
                            Or(And(Equals(ActionFunction(_ell, Set(f_one_to_i)), zero),
                                   Equals(AntiCommutator(IndexedVar(f, Add(i, one)), _ell), zero)),
                               And(NotEquals(ActionFunction(_ell, Set(f_one_to_i)), zero),
                                   InSet(Function(nu, CheckFunction(Set(f_one_to_i))),
                                         CheckFunction(Set(IndexedVar(f, Add(i, one))))))),
                            domain=Interval(one, subtract(n, one))).with_wrapping()).with_wrap_after_operator(),
              conditions=[InSet(n, Natural)]).with_wrapping(),
       domain = BufiloSets).with_wrapping(),
conditions = [IsFunction(nu, Syndromes, Detectors), Forall(A, InSet(Function(nu, A), A))]).with_wrapping()

forall_{nu : [Syndromes -> Detectors] | forall_{A} (nu(A) in A)} [forall_{b in BUFS} [exists_{f_{1}, f_{2}, ..., f_{n} | n in Natural} (({f_{1}, f_{2}, ..., f_{n}} = b) and  \\ [forall_{i in {1 .. n - 1}} (((A_{l}({f_{1}, f_{2}, ..., f_{i}}) = 0) and ({f_{i + 1}, l'} = 0)) or ((A_{l}({f_{1}, f_{2}, ..., f_{i}}) != 0) and (nu(H({f_{1}, f_{2}, ..., f_{i}})) in H({f_{i + 1}}))))])]]

A possible supporting theorem:

Given an error $e$ such that $[e, \ell]_{+} = 0$ (_i.e._, $e$ anti-commutes with $\ell$), then $e$ “contains” a fault $f$ such that $[f, \ell]_{+} = 0$ (_i.e._, $f$ also anti-commutes with $\ell$).

We _can_ think of an error $e$ as a set of faults, but we've also defined an error $e$ as an $N \times 1$ binary vector: $e \in \mathbb{F}_{2}^{N}$

In [13]:
from proveit import e, f, X
from proveit.logic import Exists, Implies
from proveit.numbers import zero
from proveit.linear_algebra import AntiCommutator
from proveit.physics.quantum.QEC2 import _ell
Implies(Equals(AntiCommutator(e, _ell), zero),
        Exists(f, Equals(AntiCommutator(f, _ell), zero), domain = e))

({e, l'} = 0) => [exists_{f in e} ({f, l'} = 0)]

#### Augmented Syndrome States and States Membership

In [14]:
from proveit.physics.quantum.QEC2 import ErrorState
state_membership_unfolding = (
    Forall(s,
           Exists(e, Equals(s, ErrorState(_ell, e)), domain = Errors),
           domain=States)
)

In [15]:
from proveit import p, G
from proveit.logic import Exists
from proveit.logic.sets import EmptySet
from proveit.graphs import Graphs, IsGraph, IsPath, Paths, Subgraph
from proveit.numbers import zero, one
from proveit.physics.quantum.QEC2 import (
        all_states_graph, AllStatesGraph, IrreducibleBufiloSets,
        Realizations, State)
exists_buf_gen_graph = Forall(b, 
       Exists(p,
              Exists(n,
                     Exists(f_one_to_n,
                            And(InSet(ExprTuple(f_one_to_n), Realizations(p, AllStatesGraph)), Equals(b, Set(f_one_to_n))),
                     domain=Errors).with_wrapping(),
              domain=Natural).with_wrapping(),
       conditions=[IsPath(p, AllStatesGraph, State(EmptySet, zero), State(EmptySet, one))]).with_wrapping(),
conditions=[LessEq(Weight(b), _max_buf_weight)], domain=IrreducibleBufiloSets).with_wrapping()

The above for “single BUFILOs” instead of all BUFILOs! Where we might use notation like $b \in 1\text{-BUF}$ to indicate that we're talking about “single BUFILOs” instead of things like triple- or 3X-BUFILOs.
Then how to define single BUFILOS? These are BUFILOs that have no BUFILOs as subsets:

$1\text{-BUF} = \{b \in \text{BUF} \,|\, \lnot \exists_{b' \in \text{BUF}} [b' \subset b]\}$

Assigned after/during discussion Fri 8/28/2026:

(1) So, we'll define the $1\text{-BUF}$ set _in addition to the original $\text{BUF}$ set, and utilize the $1\text{-BUF}$ set in the above theorem, AND go back and try to incorporate the $1\text{-BUF}$ set into Theorem 1 (originally Theorem 0);

(2) Think about the induction statement in the proof of the above. Might need a finite version of induction (see induction-related theorems in `numbers/numbersets/naturals`). We'd likely prove something along the lines of $\forall_{j \le n}$, which would then prove the statement for the existent $n \in \mathbb{N}$.

(3) Continue formulating the theorems in the paper, and edit paper to reflect the Prove-It versions of these theorems, along with some contextual text to explain.

Proof of above?

To form a BUFILO, every detector must (eventually) be turned off.

That gives us flexibility in how to order the faults!

Proof by induction:

$b = \{f_{1}, f_{2}, \ldots, f_{n}\}$

Begin: there exists $f$ such that $A_{\ell}(\{f\}) =  1$

Given that we've chosen $f_{1}, f_{2}, \ldots, f_{j}$ faults for fault sequence …

Case (1) empty syndrome? (for $j < n$) Either we've finished a BUFILO (not possible, since we can't have a BUFILO proper-subset), or we have a homologically-trivial cycle (for example, an even number of BUFILOs).

Case (2) non-empty syndrome. For all active detectors, among the remaining $(n-j)$ faults there must be at least one fault that eliminates that detector. Thus there must exist an edge e in the graph where $f_{j+1} \in EdgeFaults(e)$.

In [16]:
%end theorems

Modifying theorem mal_set_contains_non_minority_subset_of_bufilo in proveit.physics.quantum.QEC2 theory.
mal_set_contains_non_minority_subset_of_bufilo expression notebook is being updated
These theorems may now be imported from the theory package: proveit.physics.quantum.QEC2
